In [1]:
import time
import pandas as pd
import chromedriver_autoinstaller

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# =========================
# 0. 기본 설정 & 파일 경로
# =========================

INPUT_XLSX_PATH = "yeoshinticket_event_urls_part_3_of_5 (1).xlsx"
##################################################################### 각자 번호 이름 뒤에 붙여주쎄용 #######################################
# ① event_url ↔ 병원 매핑 결과 (최종)
OUTPUT_EVENT_HOSPITAL_MAP_CSV = "yeoti_event_hospital_map.csv"
OUTPUT_EVENT_HOSPITAL_MAP_XLSX = "yeoti_event_hospital_map.xlsx"

# ② 병원별 고유 리스트 (최종)
OUTPUT_HOSPITAL_UNIQUE_CSV = "yeoti_hospital_unique.csv"
OUTPUT_HOSPITAL_UNIQUE_XLSX = "yeoti_hospital_unique.xlsx"

# 테스트용: 일부만 돌리고 싶으면 숫자, 전부 돌리려면 None
MAX_ROWS = None

# 병원명 h2 기본 셀렉터 (조금 완화)
# 캡쳐 기준으로: <h2 class="font-heading20b text-gray900 cursor-pointer">
HOSPITAL_TITLE_SELECTOR_PRIMARY = "h2.font-heading20b"
HOSPITAL_TITLE_SELECTOR_FALLBACKS = [
    "section h2.font-heading20b",
    "section div.flex.items-center h2",
]

PAGE_LOAD_TIMEOUT = 10
FIND_HOSPITAL_TIMEOUT = 6
SLEEP_BETWEEN = 0.3       # 페이지 사이 딜레이 (속도 ↑)
SAVE_INTERVAL = 50         # N개 처리마다 중간 저장


# =========================
# 1. 유틸 함수 (드라이버, 저장, 진행률)
# =========================

def get_driver():
    chromedriver_autoinstaller.install()
    options = Options()
    options.add_argument("--headless=new")  # 디버깅할 땐 주석 처리
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(30)
    return driver


def print_progress(idx, total, extra_msg=""):
    pct = idx / total * 100
    msg = f"[{idx}/{total}] ({pct:5.1f}%) {extra_msg}"
    print(msg)


def save_results(event_hospital_rows, hospital_seen, final=False):
    """
    event_hospital_rows, hospital_seen을
    csv/xlsx로 저장하는 함수.
    final=False면 *_temp 파일로 중간저장.
    """
    if final:
        map_csv = OUTPUT_EVENT_HOSPITAL_MAP_CSV
        map_xlsx = OUTPUT_EVENT_HOSPITAL_MAP_XLSX
        hosp_csv = OUTPUT_HOSPITAL_UNIQUE_CSV
        hosp_xlsx = OUTPUT_HOSPITAL_UNIQUE_XLSX
        tag = "최종 저장"
    else:
        map_csv = OUTPUT_EVENT_HOSPITAL_MAP_CSV.replace(".csv", "_temp.csv")
        map_xlsx = OUTPUT_EVENT_HOSPITAL_MAP_XLSX.replace(".xlsx", "_temp.xlsx")
        hosp_csv = OUTPUT_HOSPITAL_UNIQUE_CSV.replace(".csv", "_temp.csv")
        hosp_xlsx = OUTPUT_HOSPITAL_UNIQUE_XLSX.replace(".xlsx", "_temp.xlsx")
        tag = "중간 저장"

    # (1) event ↔ 병원 매핑
    df_map = pd.DataFrame(event_hospital_rows)
    df_map.to_csv(map_csv, index=False, encoding="utf-8-sig")
    df_map.to_excel(map_xlsx, index=False)

    # (2) 병원 고유 리스트
    unique_rows = [
        {"hospital_name": name, "hospital_url": url}
        for url, name in hospital_seen.items()
    ]
    df_unique = pd.DataFrame(unique_rows)
    df_unique = df_unique.drop_duplicates(subset=["hospital_url"])

    df_unique.to_csv(hosp_csv, index=False, encoding="utf-8-sig")
    df_unique.to_excel(hosp_xlsx, index=False)

    print(f"✅ {tag} 완료 → "
          f"{map_csv}, {map_xlsx}, {hosp_csv}, {hosp_xlsx}")


# =========================
# 2. 병원 URL 추출 함수 (수정 버전)
# =========================

def _find_hospital_title_element(driver, timeout=FIND_HOSPITAL_TIMEOUT):
    """
    병원명 h2 요소를 여러 셀렉터로 탐색.
    못 찾으면 None 반환.
    """
    # 1) 페이지 중간 정도까지 스크롤
    try:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.4);")
        time.sleep(0.2)
    except Exception:
        pass

    # 2) primary 셀렉터 시도
    try:
        el = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, HOSPITAL_TITLE_SELECTOR_PRIMARY)
            )
        )
        if el.text.strip():
            return el
    except TimeoutException:
        el = None

    # 3) fallback 셀렉터들 시도
    for sel in HOSPITAL_TITLE_SELECTOR_FALLBACKS:
        try:
            candidates = driver.find_elements(By.CSS_SELECTOR, sel)
            for c in candidates:
                txt = c.text.strip()
                if txt and len(txt) <= 50:
                    return c
        except Exception:
            continue

    # 4) 그래도 못 찾으면 모든 h2 중에서 텍스트 있는 첫 번째
    try:
        h2s = driver.find_elements(By.TAG_NAME, "h2")
        for h in h2s:
            txt = h.text.strip()
            if txt and len(txt) <= 50:
                return h
    except Exception:
        pass

    return None


def get_hospital_info_from_event(driver, timeout=FIND_HOSPITAL_TIMEOUT):
    """
    현재 열려 있는 여신티켓 시술 상세페이지에서
    병원명을 클릭해 병원 URL을 얻는다.
    실패하면 (None, None) 반환.
    """
    event_url = driver.current_url

    try:
        hospital_title = _find_hospital_title_element(driver, timeout=timeout)
        if hospital_title is None:
            return None, None

        hospital_name = hospital_title.text.strip()

        # 클릭 위치로 스크롤
        try:
            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});", hospital_title
            )
            time.sleep(0.2)
        except Exception:
            pass

        # JS로 클릭 (fallback으로 .click())
        try:
            driver.execute_script("arguments[0].click();", hospital_title)
        except Exception:
            hospital_title.click()

        # URL이 바뀔 때까지 대기
        try:
            WebDriverWait(driver, timeout).until(EC.url_changes(event_url))
        except TimeoutException:
            time.sleep(0.5)

        hospital_url = driver.current_url

        if hospital_url == event_url:
            return None, None

        # 다시 시술 상세페이지로 되돌아가기
        try:
            driver.back()
            WebDriverWait(driver, timeout).until(EC.url_to_be(event_url))
        except TimeoutException:
            # 다음 루프에서 어차피 새 URL로 get() 예정
            pass

        return hospital_name, hospital_url

    except (TimeoutException, NoSuchElementException):
        return None, None


# =========================
# 3. 메인 로직
# =========================

def main():
    # --- 3-1. event_url 로드 ---
    df = pd.read_excel(INPUT_XLSX_PATH)

    if "event_url" not in df.columns:
        raise ValueError("엑셀에 'event_url' 컬럼이 없습니다.")

    event_urls = df["event_url"].dropna().astype(str).tolist()

    if MAX_ROWS is not None:
        event_urls = event_urls[:MAX_ROWS]

    total = len(event_urls)
    print(f"총 {total}개의 event_url 처리 예정")

    # --- 3-2. 드라이버 준비 ---
    driver = get_driver()

    event_hospital_rows = []  # event ↔ 병원 매핑
    hospital_seen = {}        # hospital_url 기준 dedup: {hospital_url: 병원명}

    try:
        for idx, url in enumerate(event_urls, start=1):
            print_progress(idx, total, extra_msg=url)

            try:
                driver.get(url)
            except Exception as e:
                print(f"  ⚠ 페이지 로딩 실패: {e}")
                event_hospital_rows.append({
                    "event_url": url,
                    "hospital_name": None,
                    "hospital_url": None,
                    "status": "page_load_error"
                })
                time.sleep(SLEEP_BETWEEN)
                # 중간 저장 체크
                if idx % SAVE_INTERVAL == 0:
                    save_results(event_hospital_rows, hospital_seen, final=False)
                continue

            # 시술 페이지 로딩 확인 (h1 하나만 체크)
            try:
                WebDriverWait(driver, PAGE_LOAD_TIMEOUT).until(
                    EC.presence_of_element_located((By.TAG_NAME, "h1"))
                )
            except TimeoutException:
                print("  ⚠ 시술 페이지 로딩 타임아웃")
                event_hospital_rows.append({
                    "event_url": url,
                    "hospital_name": None,
                    "hospital_url": None,
                    "status": "event_page_timeout"
                })
                time.sleep(SLEEP_BETWEEN)
                if idx % SAVE_INTERVAL == 0:
                    save_results(event_hospital_rows, hospital_seen, final=False)
                continue

            # 병원 정보 추출
            hospital_name, hospital_url = get_hospital_info_from_event(
                driver, timeout=FIND_HOSPITAL_TIMEOUT
            )

            status = "success" if hospital_url else "no_hospital_url"
            print(f"  → 병원명: {hospital_name}, 병원URL: {hospital_url}, status: {status}")

            # event ↔ 병원 기록
            event_hospital_rows.append({
                "event_url": url,
                "hospital_name": hospital_name,
                "hospital_url": hospital_url,
                "status": status
            })

            # 병원 URL 기준 dedup
            if hospital_url:
                if hospital_url not in hospital_seen:
                    hospital_seen[hospital_url] = hospital_name

            time.sleep(SLEEP_BETWEEN)

            # N개마다 중간 저장
            if idx % SAVE_INTERVAL == 0:
                save_results(event_hospital_rows, hospital_seen, final=False)

    finally:
        driver.quit()

    # --- 3-3. 최종 결과 저장 ---
    save_results(event_hospital_rows, hospital_seen, final=True)


if __name__ == "__main__":
    main()


총 924개의 event_url 처리 예정
[1/924] (  0.1%) https://www.yeoshin.co.kr/event/mobile/15644
  → 병원명: 동안중심의원, 병원URL: https://www.yeoshin.co.kr/hospitals/4986?from=events&tabmenu=EVENTS, status: success
[2/924] (  0.2%) https://www.yeoshin.co.kr/event/mobile/26348
  → 병원명: 새라한의원, 병원URL: https://www.yeoshin.co.kr/hospitals/86281?from=events&tabmenu=EVENTS, status: success
[3/924] (  0.3%) https://www.yeoshin.co.kr/event/mobile/14773
  → 병원명: 영드림의원, 병원URL: https://www.yeoshin.co.kr/hospitals/4493?from=events&tabmenu=EVENTS, status: success
[4/924] (  0.4%) https://www.yeoshin.co.kr/event/mobile/24645
  → 병원명: 더이뻐의원, 병원URL: https://www.yeoshin.co.kr/hospitals/103832?from=events&tabmenu=EVENTS, status: success
[5/924] (  0.5%) https://www.yeoshin.co.kr/event/mobile/16179
  → 병원명: 제너리스의원(연신내점), 병원URL: https://www.yeoshin.co.kr/hospitals/5011?from=events&tabmenu=EVENTS, status: success
[6/924] (  0.6%) https://www.yeoshin.co.kr/event/mobile/23085
  → 병원명: 뷰티블라썸의원, 병원URL: https://www.yeoshin.co.kr/ho